In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv("./data/base_clean.csv")

/tmp/ipykernel_6560/1050980111.py:1: DtypeWarning: Columns (20: first_chart_week, 23: charted_before_2000, 24: still_charting) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./data/base_clean.csv")


In [4]:
lyr = df[["id", "lyrics", "language_ft"]]

# Cleaning

Strip [] in lyrics

In [5]:
lyr["lyrics"] = (
    df["lyrics"]
    .str.replace(r"\[.*?\]", "", regex=True, flags=re.DOTALL)
    .str.replace(r"\n{2,}", "\n", regex=True)
    .str.strip()
)

Keep only "language_ft" = "en"

In [6]:
lyr = lyr[lyr["language_ft"]=="en"]

# Vocabulary

In [7]:
clean = (
    df["lyrics"]
    .str.lower()
    .str.replace(r"[^\w\s']", "", regex=True)
)

Word count

In [8]:
lyr["word_count"] = lyr["lyrics"].str.split().str.len()
lyr["unique_count"] = df["lyrics"].str.lower().str.split().apply(lambda w: len(set(w)) if  isinstance(w, list) else 0)

## Type-Token Ratio (TTR): unique words / total words

In [9]:
lyr["TTR"] = lyr["unique_count"] / lyr["word_count"]


Moving average TTR: Slide a window across the lyrics, computes TTR per window, and averages.

In [10]:
def mattr(text, window=50):
    if not isinstance(text, str):
        return float("nan")
    words = text.split()
    if len(words) < window:
        return float("nan")

    ttrs = [len(set(words[i:i+window])) / window for i in range(len(words) - window +1)]
    return sum(ttrs) / len(ttrs)

## Yule's K

K = 10000 * (\sum(f^2(i)) - word_count) / word_count^2

lower k_value means more diverse vocab and vice versa

In [11]:
from collections import Counter

def yules_k(text):
    if not isinstance(text, str):
        return float("nan")

    words = text.split()
    N = len(words)
    if N == 0:
        return float("nan")
    freqs = Counter(words).values()
    s2 = sum(f * f for f in freqs)
    return 10000 * (s2 - N) / (N**2)

lyr["yules_k"] = clean.apply(yules_k)

## Profanity

In [15]:
from better_profanity import profanity
from pandarallel import pandarallel

pandarallel.initialize()

profanity.load_censor_words()
PROFANE = frozenset(str(w).lower() for w in profanity.CENSOR_WORDSET)

def count_profanity(text):
    if not isinstance(text, str):
        return float("nan")
    return sum(w in PROFANE for w in text.split())

lyr["profanity_count"] = clean.parallel_apply(count_profanity)

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [16]:
lyr

,id,lyrics,language_ft,word_count,unique_count,TTR,yules_k,profanity_count
0,5A6OHHy73AR5tLxgTc98zz,"Yeah, uh-huh, you know what it is\nBlack and y...",en,656,200,0.304878,260.362509,8.0
1,0O45fw2L5vsWpdsOdXwNAR,I'm bringin' sexy back (Yeah)\nThem other boys...,en,673,126,0.187221,347.590016,4.0
2,2zZJnAlhgvW6DERn1tu3fb,"Ooh\nOoh, mmm, mmm\nHave you ever reached a ra...",en,389,156,0.401028,183.083439,1.0
3,0sf12qNH5qcw8qpgymFOqD,Yeah\nI've been tryna call\nI've been on my ow...,en,259,115,0.444015,178.646805,0.0
4,7wZUrN8oemZfsEd1CGkbXE,"Closed off from love, I didn't need the pain\n...",en,379,126,0.332454,291.468528,0.0
...,...,...,...,...,...,...,...,...
160195,1fXmDeiCb3ABt5CzkMxp4u,Yeah\nI saw you movin' on the dance floor\nI c...,en,371,148,0.398922,128.826615,0.0
160196,6sHxpnCerm9M6A5zaFFNfs,I'm sitting here cold as ice\nSwallow my frien...,en,320,118,0.368750,138.970719,3.0
160199,27kcZEJvhkb1rzZS9gCpdA,I've been rememberin' the mornings\nWhere we'd...,en,383,136,0.355091,151.124739,0.0
160204,0NuWgxEp51CutD2pJoF4OM,"She got the gold hoops, Prada fit\nI'm in love...",en,267,114,0.426966,153.266692,0.0


# Theme